# Hybrid Search in LlamaIndex

Hybrid search combines keyword search (great at exact terms) with vector search (great at meaning) so neither approach's blind spots dominate the final result.


Standard setup — quiet logging (including the BM25 library used below), load env vars, and set the default LLM/embedding model.


In [1]:
import logging

from dotenv import load_dotenv
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

# Quiet noisy INFO-level logs from LlamaIndex, its HTTP client, and the BM25
# library we'll use later for hybrid search.
for noisy_logger in ("httpx", "llama_index", "bm25s"):
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)

# Load API keys from .env into the environment.
load_dotenv()

# Set the default LLM and embedding model used everywhere in this notebook.
Settings.llm = OpenAI(model="gpt-4.1-nano")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

**Step 1 — Build the baseline index.** Load the anime corpus and build a `VectorStoreIndex`, exactly as in earlier episodes — the vector half of the hybrid retriever we build next queries this same index.


In [2]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

# Load the anime corpus and build a VectorStoreIndex from it — the vector half
# of the hybrid retriever we build next queries this same index.
documents = SimpleDirectoryReader("data/sample_docs").load_data()
index = VectorStoreIndex.from_documents(documents)
print(f"Index ready with {len(index.docstore.docs)} nodes")

Index ready with 8 nodes


**Step 2 — Combine keyword and vector search.** Build a hybrid retriever that fuses a BM25 keyword retriever with the existing vector retriever via reciprocal rank fusion, then compare it against vector-only search on a question with a very exact, keyword-friendly answer.


In [3]:
# Some hosted vector stores (Weaviate, Qdrant, Pinecone, etc.) support hybrid
# search natively, server-side. Here we build it locally at no extra cost by
# fusing a keyword retriever (BM25) with our vector retriever via reciprocal
# rank fusion — no external infrastructure required.
import logging

import Stemmer
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.retrievers.bm25 import BM25Retriever

# bm25s sets its own logger to DEBUG on import, overriding the suppression above
logging.getLogger("bm25s").setLevel(logging.WARNING)

vector_retriever = index.as_retriever(similarity_top_k=3)  # semantic similarity search
bm25_retriever = BM25Retriever.from_defaults(
    docstore=index.docstore, similarity_top_k=3, stemmer=Stemmer.Stemmer("english")
)  # classic keyword search over the same nodes

hybrid_retriever = QueryFusionRetriever(
    [vector_retriever, bm25_retriever],
    similarity_top_k=3,
    num_queries=1,  # skip LLM query rewriting, fuse results for this exact question
    mode="reciprocal_rerank",  # merge both retrievers' rankings into one combined ranking
)
hybrid_engine = RetrieverQueryEngine.from_args(hybrid_retriever)

keyword_question = "Which arc introduces the Room of Spirit and Time where a day outside equals a year inside?"
vector_only_response = index.as_query_engine(similarity_top_k=3).query(keyword_question)
hybrid_response = hybrid_engine.query(keyword_question)

print("--- Vector-only ---")
print(vector_only_response)
print("\n--- Hybrid (vector + BM25 keyword) ---")
print(hybrid_response)

--- Vector-only ---
The provided context does not include information about an arc that introduces the Room of Spirit and Time where a day outside equals a year inside.

--- Hybrid (vector + BM25 keyword) ---
The arc that introduces the Room of Spirit and Time, where a day outside equals a year inside, is the Cell Saga.


On a tiny 5-document demo corpus like this one, vector-only and hybrid search often agree, since there just aren't many competing chunks to disambiguate between. Hybrid search earns its keep at real-world scale — hundreds or thousands of documents — where exact terms like character names, numbers, or rule specifics can get diluted among many semantically-similar-but-wrong chunks.


### Summary

- Hybrid search combines keyword precision (BM25) with vector recall (semantic similarity) via reciprocal rank fusion — you don't need a specialized vector store to get real hybrid search, though some hosted stores do it natively server-side.
